<a href="https://colab.research.google.com/github/rounak393/clab/blob/main/lovasz_hinge_loss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


# !pip install albumentations scikit-learn -q

import os
import cv2
import glob
import copy
import random
import warnings
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

from sklearn.model_selection import train_test_split, StratifiedKFold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet34, ResNet34_Weights
import albumentations as A

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =============================================================================
# 1. CONFIGURATION
# =============================================================================
BATCH_SIZE   = 8
EPOCHS       = 100
LR           = 1e-4
IMG_SIZE     = 256
SEED         = 42
N_SPLITS     = 5
TEST_SIZE    = 0.15
WARMUP_EP    = 10           # linear warmup epochs
LABEL_SMOOTH = 0.05         # label smoothing epsilon
CLASSES      = ["benign", "malignant"]
SAVE_DIR     = "/kaggle/working"
THRESHOLDS   = [0.35, 0.40, 0.45, 0.50, 0.55, 0.60]   # tuning candidates
os.makedirs(SAVE_DIR, exist_ok=True)

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.benchmark     = True
torch.backends.cudnn.deterministic = True

# =============================================================================
# 2. AUGMENTATIONS
# =============================================================================
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(
        shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5,
        border_mode=cv2.BORDER_CONSTANT, value=0
    ),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussNoise(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

val_test_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

# =============================================================================
# 3. DATASET
# =============================================================================
class BUSIDataset(Dataset):
    def __init__(self, base_dir, classes, transform=None, indices=None):
        self.transform    = transform
        self._all_samples = []

        for cls in classes:
            cls_dir = os.path.join(base_dir, cls)
            if not os.path.exists(cls_dir):
                continue
            images = sorted([
                f for f in os.listdir(cls_dir)
                if f.endswith(".png") and "_mask" not in f
            ])
            label = 0 if cls == "benign" else 1
            for img_name in images:
                img_path   = os.path.join(cls_dir, img_name)
                base_name  = img_name.replace(".png", "")
                mask_files = sorted([
                    f for f in os.listdir(cls_dir)
                    if f.startswith(base_name + "_mask") and f.endswith(".png")
                ])
                if not mask_files:
                    continue
                self._all_samples.append((
                    img_path,
                    [os.path.join(cls_dir, f) for f in mask_files],
                    label
                ))

        if indices is not None:
            self._all_samples = [self._all_samples[i] for i in indices]

    def __len__(self):
        return len(self._all_samples)

    def __getitem__(self, idx):
        img_path, mask_paths, _ = self._all_samples[idx]

        image = np.array(Image.open(img_path).convert("RGB"))

        combined_mask = np.zeros(image.shape[:2], dtype=np.uint8)
        for mpath in mask_paths:
            m = np.array(Image.open(mpath).convert("L"))
            combined_mask = np.logical_or(combined_mask, m > 0).astype(np.uint8)
        combined_mask = combined_mask.astype(np.float32)

        if self.transform:
            aug = self.transform(image=image, mask=combined_mask)
            image, combined_mask = aug["image"], aug["mask"]

        combined_mask = (combined_mask > 0.5).astype(np.float32)

        return (torch.from_numpy(image).permute(2, 0, 1).float(),
                torch.from_numpy(combined_mask).unsqueeze(0).float())

    @property
    def labels(self):
        return [s[2] for s in self._all_samples]



class RobertsEdgeOperator(nn.Module):
    def __init__(self):
        super().__init__()
        kx = torch.tensor([[[[1.0, 0.0], [0.0, -1.0]]]])
        ky = torch.tensor([[[[0.0, 1.0], [-1.0, 0.0]]]])
        self.register_buffer("kx", kx)
        self.register_buffer("ky", ky)

    def forward(self, x):
        gray = (0.2989 * x[:, 0:1]
               + 0.5870 * x[:, 1:2]
               + 0.1140 * x[:, 2:3])
        gp = F.pad(gray, (0, 1, 0, 1), mode="replicate")
        return torch.sqrt(
            F.conv2d(gp, self.kx) ** 2 +
            F.conv2d(gp, self.ky) ** 2 + 1e-8)


class PCBAM_Filter(nn.Module):

    def __init__(self, in_channels, reduction=8):
        super().__init__()
        mid = max(in_channels // reduction, 4)
        self.cam = nn.Sequential(
            nn.Linear(in_channels, mid),
            nn.ReLU(inplace=True),
            nn.Linear(mid, in_channels)
        )
        self.sam = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)

    def forward(self, x):
        b, c, _, _ = x.size()
        avg = self.cam(
            F.adaptive_avg_pool2d(x, 1).view(b, c)).view(b, c, 1, 1)
        mxp = self.cam(
            F.adaptive_max_pool2d(x, 1).view(b, c)).view(b, c, 1, 1)
        x_c = x * torch.sigmoid(avg + mxp)
        sp  = torch.cat([
            torch.mean(x_c, dim=1, keepdim=True),
            torch.max(x_c,  dim=1, keepdim=True)[0]
        ], dim=1)
        return x_c * torch.sigmoid(self.sam(sp))


class SpatialGateAttention(nn.Module):

    def __init__(self, in_channels):
        super().__init__()
        mid = in_channels // 8
        self.q_conv = nn.Conv2d(in_channels, mid, 1)
        self.k_conv = nn.Conv2d(in_channels, mid, 1)
        self.v_conv = nn.Conv2d(in_channels, in_channels, 1)
        self.gate   = nn.Conv2d(mid * 2 + in_channels, in_channels, 1)

    def forward(self, x):
        q = self.q_conv(x)
        k = self.k_conv(x)
        v = self.v_conv(x)
        A = torch.sigmoid(self.gate(torch.cat([q, k, v], dim=1)))
        return x + v * A


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class MaxDiceUNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.roberts = RobertsEdgeOperator()

        resnet   = resnet34(weights=ResNet34_Weights.IMAGENET1K_V1)
        old_conv = resnet.conv1
        self.conv1 = nn.Conv2d(4, 64, kernel_size=7, stride=2, padding=3,
                               bias=False)
        with torch.no_grad():
            self.conv1.weight[:, :3] = old_conv.weight
            self.conv1.weight[:, 3]  = old_conv.weight.mean(dim=1)

        self.bn1     = resnet.bn1
        self.relu    = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1  = resnet.layer1   # 64ch
        self.layer2  = resnet.layer2   # 128ch
        self.layer3  = resnet.layer3   # 256ch
        self.layer4  = resnet.layer4   # 512ch

        self.pcbam1 = PCBAM_Filter(64)
        self.pcbam2 = PCBAM_Filter(128)
        self.pcbam3 = PCBAM_Filter(256)
        self.pcbam4 = PCBAM_Filter(512)

        self.pool       = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(512, 1024)
        self.sga        = SpatialGateAttention(1024)

        self.up4  = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = DoubleConv(1024, 512)

        self.up3  = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = DoubleConv(512, 256)

        self.up2  = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = DoubleConv(256, 128)

        self.up1  = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        self.up0  = nn.ConvTranspose2d(64, 64, 2, stride=2)
        self.dec0 = DoubleConv(128, 64)

        self.up_out  = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec_out = DoubleConv(32, 32)
        self.final   = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        x_edge  = self.roberts(x)
        x_fused = torch.cat([x, x_edge], dim=1)

        x0 = self.relu(self.bn1(self.conv1(x_fused)))
        x1 = self.maxpool(x0)

        e1 = self.layer1(x1)
        e2 = self.layer2(e1)
        e3 = self.layer3(e2)
        e4 = self.layer4(e3)

        s1 = self.pcbam1(e1)
        s2 = self.pcbam2(e2)
        s3 = self.pcbam3(e3)
        s4 = self.pcbam4(e4)

        b  = self.sga(self.bottleneck(self.pool(e4)))

        d4 = self.dec4(torch.cat([self.up4(b),  s4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), s3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), s2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), s1], dim=1))
        d0 = self.dec0(torch.cat([self.up0(d1), x0], dim=1))

        out = self.dec_out(self.up_out(d0))
        return self.final(out)

# =============================================================================
# 6. LOVÁSZ LOSS + HYBRID LOSS
# =============================================================================

def lovasz_grad(gt_sorted):

    p            = len(gt_sorted)
    gts          = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union        = gts + (1 - gt_sorted).float().cumsum(0)
    jaccard      = 1.0 - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard


def lovasz_hinge(logits, labels):

    logits = logits.view(-1)
    labels = labels.view(-1).float()
    signs  = 2.0 * labels - 1.0
    errors = 1.0 - logits * signs
    errors_sorted, perm = torch.sort(errors, descending=True)
    gt_sorted = labels[perm.data]
    grad      = lovasz_grad(gt_sorted)
    return torch.dot(F.relu(errors_sorted), grad)


class HybridLoss(nn.Module):

    def __init__(self, smooth=1e-5, focal_gamma=2.0,
                 label_smooth=LABEL_SMOOTH):
        super().__init__()
        self.smooth       = smooth
        self.focal_gamma  = focal_gamma
        self.label_smooth = label_smooth

    def forward(self, logits, targets):
        # Label-smoothed BCE
        targets_smooth = targets * (1 - self.label_smooth) + 0.5 * self.label_smooth
        bce = F.binary_cross_entropy_with_logits(logits, targets_smooth)

        # Dice
        probs = torch.sigmoid(logits).view(-1)
        tgt   = targets.view(-1)
        inter = (probs * tgt).sum()
        dice  = 1.0 - (2.0 * inter + self.smooth) / (
                    probs.sum() + tgt.sum() + self.smooth)

        # Focal
        pt    = torch.where(tgt == 1, probs, 1.0 - probs)
        focal = (-(1.0 - pt) ** self.focal_gamma
                 * torch.log(pt + 1e-8)).mean()

        # Lovász-Hinge
        lovasz = lovasz_hinge(logits, targets)

        return 0.25 * bce + 0.25 * dice + 0.15 * focal + 0.35 * lovasz

# =============================================================================
# 7. METRICS
# =============================================================================

def compute_metrics(logits, targets, threshold=0.5, smooth=1e-5):
    """Dice, IoU, Precision, Recall, F1 from raw logits."""
    preds = (torch.sigmoid(logits) > threshold).float().view(-1)
    tgts  = targets.view(-1).float()
    tp = (preds * tgts).sum()
    fp = (preds * (1 - tgts)).sum()
    fn = ((1 - preds) * tgts).sum()
    precision = (tp + smooth) / (tp + fp + smooth)
    recall    = (tp + smooth) / (tp + fn + smooth)
    f1        = (2 * precision * recall) / (precision + recall + smooth)
    dice      = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
    iou       = (tp + smooth) / (tp + fp + fn + smooth)
    return {
        "dice":      dice.item(),
        "iou":       iou.item(),
        "precision": precision.item(),
        "recall":    recall.item(),
        "f1":        f1.item()
    }


def compute_metrics_from_probs(probs, targets, threshold=0.5, smooth=1e-5):
    """Same as compute_metrics but accepts probabilities directly."""
    preds = (probs > threshold).float().view(-1)
    tgts  = targets.view(-1).float()
    tp = (preds * tgts).sum()
    fp = (preds * (1 - tgts)).sum()
    fn = ((1 - preds) * tgts).sum()
    precision = (tp + smooth) / (tp + fp + smooth)
    recall    = (tp + smooth) / (tp + fn + smooth)
    f1        = (2 * precision * recall) / (precision + recall + smooth)
    dice      = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
    iou       = (tp + smooth) / (tp + fp + fn + smooth)
    return {
        "dice":      dice.item(),
        "iou":       iou.item(),
        "precision": precision.item(),
        "recall":    recall.item(),
        "f1":        f1.item()
    }


def aggregate_metrics(metric_list):
    keys = metric_list[0].keys()
    return {k: float(np.mean([m[k] for m in metric_list])) for k in keys}

# =============================================================================
# 8. LR SCHEDULER — Linear Warmup + Cosine Decay
# =============================================================================

def build_scheduler(optimizer, warmup_epochs, total_epochs):

    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return float(epoch + 1) / float(warmup_epochs)
        progress = float(epoch - warmup_epochs) / float(
                   max(1, total_epochs - warmup_epochs))
        return max(1e-6 / LR, 0.5 * (1.0 + np.cos(np.pi * progress)))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

#

def predict_tta(model, image_tensor):

    with torch.amp.autocast("cuda"):
        p1 = torch.sigmoid(model(image_tensor))

        p2 = torch.sigmoid(model(torch.flip(image_tensor, dims=[3])))
        p2 = torch.flip(p2, dims=[3])

        p3 = torch.sigmoid(model(torch.flip(image_tensor, dims=[2])))
        p3 = torch.flip(p3, dims=[2])

        p4 = torch.sigmoid(model(torch.rot90(image_tensor, k=1, dims=[2, 3])))
        p4 = torch.rot90(p4, k=3, dims=[2, 3])

        p5 = torch.sigmoid(model(torch.rot90(image_tensor, k=3, dims=[2, 3])))
        p5 = torch.rot90(p5, k=1, dims=[2, 3])

    return (p1 + p2 + p3 + p4 + p5) / 5.0

# =============================================================================
# 10. DATA SPLITTING
# =============================================================================

possible_paths = glob.glob("/kaggle/input/**/benign", recursive=True)
if not possible_paths:
    raise FileNotFoundError("Dataset not found.")
BASE_DIR     = os.path.dirname(possible_paths[0])
full_dataset = BUSIDataset(BASE_DIR, classes=CLASSES, transform=None)

all_indices = np.arange(len(full_dataset))
all_labels  = np.array(full_dataset.labels)

print(f"Total samples : {len(all_indices)}")
print(f"  Benign      : {(all_labels == 0).sum()}")
print(f"  Malignant   : {(all_labels == 1).sum()}")

trainval_idx, test_idx = train_test_split(
    all_indices, test_size=TEST_SIZE,
    stratify=all_labels, random_state=SEED
)
trainval_labels = all_labels[trainval_idx]
test_labels     = all_labels[test_idx]

print(f"\nTrain+Val pool    : {len(trainval_idx)}")
print(f"Test set (locked) : {len(test_idx)}")
print(f"  Test benign     : {(test_labels == 0).sum()}")
print(f"  Test malignant  : {(test_labels == 1).sum()}")



skf              = StratifiedKFold(n_splits=N_SPLITS, shuffle=True,
                                   random_state=SEED)
criterion        = HybridLoss()
fold_paths       = []
fold_val_metrics = []
fold_thresholds  = []   # best threshold per fold from tuning

print(f"\n STARTING {N_SPLITS}-FOLD STRATIFIED CROSS VALIDATION")
print("=" * 65)

for fold, (rel_train_idx, rel_val_idx) in enumerate(
        skf.split(trainval_idx, trainval_labels)):

    abs_train_idx = trainval_idx[rel_train_idx]
    abs_val_idx   = trainval_idx[rel_val_idx]

    print(f"\n--- FOLD {fold+1}/{N_SPLITS} ---")
    print(f"  Train: {len(abs_train_idx)} | Val: {len(abs_val_idx)}")

    train_ds = BUSIDataset(BASE_DIR, CLASSES, train_transform,
                           indices=abs_train_idx)
    val_ds   = BUSIDataset(BASE_DIR, CLASSES, val_test_transform,
                           indices=abs_val_idx)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)

    model     = MaxDiceUNet().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = build_scheduler(optimizer, WARMUP_EP, EPOCHS)
    scaler    = torch.amp.GradScaler("cuda")

    best_val_dice = 0.0
    best_weights  = None
    history       = {"train_dice": [], "val_dice": [], "val_iou": []}

    for epoch in range(EPOCHS):

        # ── Training ─────────────────────────────────────────────────────
        model.train()
        epoch_train_metrics = []

        for images, masks in tqdm(train_loader,
                                  desc=f"F{fold+1} E{epoch+1:03d}",
                                  leave=False):
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda"):
                logits = model(images)
                loss   = criterion(logits, masks)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            with torch.no_grad():
                epoch_train_metrics.append(compute_metrics(logits, masks))

        scheduler.step()

        # ── Validation
        model.eval()
        val_metric_list = []

        with torch.no_grad():
            for images, masks in val_loader:
                images, masks = images.to(device), masks.to(device)
                with torch.amp.autocast("cuda"):
                    logits = model(images)
                val_metric_list.append(compute_metrics(logits, masks))

        avg_train = aggregate_metrics(epoch_train_metrics)
        avg_val   = aggregate_metrics(val_metric_list)

        history["train_dice"].append(avg_train["dice"])
        history["val_dice"].append(avg_val["dice"])
        history["val_iou"].append(avg_val["iou"])

        if avg_val["dice"] > best_val_dice:
            best_val_dice = avg_val["dice"]
            best_weights  = copy.deepcopy(model.state_dict())
            save_path     = os.path.join(
                SAVE_DIR, f"best_model_fold_{fold+1}.pth")
            torch.save(best_weights, save_path)

    fold_paths.append(save_path)
    fold_val_metrics.append(best_val_dice)


    model.load_state_dict(torch.load(save_path, weights_only=True))
    model.eval()

    all_probs_val  = []
    all_masks_val  = []

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            with torch.amp.autocast("cuda"):
                logits = model(images)
            all_probs_val.append(torch.sigmoid(logits).cpu())
            all_masks_val.append(masks)

    all_probs_val = torch.cat(all_probs_val, dim=0)
    all_masks_val = torch.cat(all_masks_val, dim=0)

    print(f"\n  Threshold tuning on val set (fold {fold+1}):")
    best_t    = 0.5
    best_t_dice = 0.0

    for t in THRESHOLDS:
        m = compute_metrics_from_probs(all_probs_val, all_masks_val,
                                       threshold=t)
        flag = " ← best" if m["dice"] > best_t_dice else ""
        print(f"    t={t:.2f}: Dice={m['dice']:.4f}  "
              f"IoU={m['iou']:.4f}  "
              f"Prec={m['precision']:.4f}  "
              f"Rec={m['recall']:.4f}{flag}")
        if m["dice"] > best_t_dice:
            best_t_dice = m["dice"]
            best_t      = t

    fold_thresholds.append(best_t)
    print(f"  Best threshold for fold {fold+1}: {best_t}")
    print(f" Fold {fold+1} | Best Val Dice: {best_val_dice:.4f} "
          f"| Best Threshold: {best_t}")

    # ── Training curve ────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(history["train_dice"], label="Train Dice", linewidth=1.5)
    ax.plot(history["val_dice"],   label="Val Dice",   linewidth=1.5)
    ax.plot(history["val_iou"],    label="Val IoU",
            linewidth=1.5, linestyle="--")
    ax.axvline(WARMUP_EP, color="gray", linestyle=":",
               alpha=0.7, label=f"End warmup (ep {WARMUP_EP})")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Score")
    ax.set_title(f"Fold {fold+1} — MaxDiceUNet v4 (Lovász + Warmup)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f"curve_fold_{fold+1}.png"), dpi=150)
    plt.close()

print("\n Cross-Validation Summary")
print("=" * 65)
for i, (d, t) in enumerate(zip(fold_val_metrics, fold_thresholds)):
    print(f"  Fold {i+1}: Dice={d:.4f}  Best threshold={t}")
print(f"\n  Mean ± Std : {np.mean(fold_val_metrics):.4f} "
      f"± {np.std(fold_val_metrics):.4f}")

from collections import Counter
threshold_votes = Counter(fold_thresholds)
BEST_THRESHOLD  = threshold_votes.most_common(1)[0][0]
print(f"  Threshold votes : {dict(threshold_votes)}")
print(f"  Final threshold (majority vote) : {BEST_THRESHOLD}")


test_ds     = BUSIDataset(BASE_DIR, CLASSES, val_test_transform,
                          indices=test_idx)
test_loader = DataLoader(test_ds, batch_size=4, shuffle=False,
                         num_workers=2, pin_memory=True)

ensemble = []
for path in fold_paths:
    m = MaxDiceUNet().to(device)
    m.load_state_dict(torch.load(path, weights_only=True))
    m.eval()
    ensemble.append(m)

print("\n FINAL EVALUATION ON HELD-OUT TEST SET")
print("=" * 65)

configs = {
    "Single model (fold 1), t=0.50, no TTA":
        (ensemble[:1], False, 0.50),
    "Ensemble (5 models),   t=0.50, no TTA":
        (ensemble,     False, 0.50),
    f"Ensemble (5 models), t={BEST_THRESHOLD:.2f},  5-TTA":
        (ensemble,     True,  BEST_THRESHOLD),
}

results_table = {}

for config_name, (models, use_tta, threshold) in configs.items():
    all_metrics = []

    with torch.no_grad():
        for images, masks in tqdm(test_loader, desc=config_name):
            images = images.to(device)
            masks  = masks.to(device)

            avg_probs = torch.zeros_like(masks)
            for m in models:
                if use_tta:
                    avg_probs += predict_tta(m, images)
                else:
                    with torch.amp.autocast("cuda"):
                        avg_probs += torch.sigmoid(m(images))
            avg_probs /= len(models)

            all_metrics.append(
                compute_metrics_from_probs(avg_probs, masks,
                                           threshold=threshold))

    agg = aggregate_metrics(all_metrics)
    results_table[config_name] = agg

    print(f"\n  {config_name}")
    for k, v in agg.items():
        print(f"    {k:<12}: {v:.4f}")


print("\n\n ABLATION TABLE — MaxDiceUNet v4")
print("=" * 82)
print(f"{'Configuration':<44} {'Dice':>6} {'IoU':>6} "
      f"{'Prec':>6} {'Rec':>6} {'F1':>6}")
print("-" * 82)
for name, m in results_table.items():
    print(f"{name:<44} {m['dice']:>6.4f} {m['iou']:>6.4f} "
          f"{m['precision']:>6.4f} {m['recall']:>6.4f} {m['f1']:>6.4f}")


final_key = f"Ensemble (5 models), t={BEST_THRESHOLD:.2f},  5-TTA"
final     = results_table[final_key]
print("\n\n PAPER REPORTING NUMBERS")
print("=" * 65)
print(f"  CV  Dice  : {np.mean(fold_val_metrics):.4f} "
      f"± {np.std(fold_val_metrics):.4f}")
print(f"  Test Dice : {final['dice']:.4f}")
print(f"  Test IoU  : {final['iou']:.4f}")
print(f"  Test F1   : {final['f1']:.4f}")
print(f"  Threshold : {BEST_THRESHOLD}")


print("\n Saving sample predictions...")

mean_ = np.array([0.485, 0.456, 0.406])
std_  = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(4, 3, figsize=(10, 14))
for j, t in enumerate(["Input Image", "Ground Truth",
                        f"Prediction (Ensemble+5TTA, t={BEST_THRESHOLD})"]):
    axes[0, j].set_title(t, fontsize=10, fontweight="bold")

with torch.no_grad():
    sample_imgs, sample_masks = next(iter(test_loader))
    imgs_gpu = sample_imgs.to(device)

    avg_p = torch.zeros(
        sample_imgs.shape[0], 1, IMG_SIZE, IMG_SIZE).to(device)
    for m in ensemble:
        avg_p += predict_tta(m, imgs_gpu)
    avg_p    /= len(ensemble)
    preds_bin = (avg_p > BEST_THRESHOLD).float().cpu().numpy()

for i in range(min(4, len(sample_imgs))):
    img_np = np.clip(
        sample_imgs[i].permute(1, 2, 0).numpy() * std_ + mean_, 0, 1)
    axes[i, 0].imshow(img_np);              axes[i, 0].axis("off")
    axes[i, 1].imshow(sample_masks[i, 0].numpy(), cmap="gray")
    axes[i, 1].axis("off")
    axes[i, 2].imshow(preds_bin[i, 0], cmap="gray")
    axes[i, 2].axis("off")

plt.suptitle(
    f"MaxDiceUNet v4 — Lovász+Warmup — Test Set (t={BEST_THRESHOLD})",
    fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "sample_predictions.png"),
            dpi=150, bbox_inches="tight")
plt.close()

print(f"\n All outputs saved to: {SAVE_DIR}")
print("  Files: best_model_fold_1..5.pth | curve_fold_1..5.png "
      "| sample_predictions.png")

Using device: cuda
Total samples : 647
  Benign      : 437
  Malignant   : 210

Train+Val pool    : 549
Test set (locked) : 98
  Test benign     : 66
  Test malignant  : 32

 STARTING 5-FOLD STRATIFIED CROSS VALIDATION

--- FOLD 1/5 ---
  Train: 439 | Val: 110
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 246MB/s]



  Threshold tuning on val set (fold 1):
    t=0.35: Dice=0.8071  IoU=0.6765  Prec=0.8189  Rec=0.7955 ← best
    t=0.40: Dice=0.8080  IoU=0.6779  Prec=0.8283  Rec=0.7887 ← best
    t=0.45: Dice=0.8085  IoU=0.6785  Prec=0.8357  Rec=0.7830 ← best
    t=0.50: Dice=0.8085  IoU=0.6786  Prec=0.8422  Rec=0.7774 ← best
    t=0.55: Dice=0.8082  IoU=0.6781  Prec=0.8483  Rec=0.7717
    t=0.60: Dice=0.8074  IoU=0.6770  Prec=0.8544  Rec=0.7653
  Best threshold for fold 1: 0.5
 Fold 1 | Best Val Dice: 0.8078 | Best Threshold: 0.5

--- FOLD 2/5 ---
  Train: 439 | Val: 110



  Threshold tuning on val set (fold 2):
    t=0.35: Dice=0.7983  IoU=0.6643  Prec=0.7707  Rec=0.8280 ← best
    t=0.40: Dice=0.8016  IoU=0.6689  Prec=0.7845  Rec=0.8194 ← best
    t=0.45: Dice=0.8039  IoU=0.6721  Prec=0.7971  Rec=0.8108 ← best
    t=0.50: Dice=0.8056  IoU=0.6745  Prec=0.8093  Rec=0.8020 ← best
    t=0.55: Dice=0.8068  IoU=0.6761  Prec=0.8214  Rec=0.7927 ← best
    t=0.60: Dice=0.8071  IoU=0.6767  Prec=0.8351  Rec=0.7810 ← best
  Best threshold for fold 2: 0.6
 Fold 2 | Best Val Dice: 0.8074 | Best Threshold: 0.6

--- FOLD 3/5 ---
  Train: 439 | Val: 110



  Threshold tuning on val set (fold 3):
    t=0.35: Dice=0.8030  IoU=0.6708  Prec=0.8117  Rec=0.7944 ← best
    t=0.40: Dice=0.8049  IoU=0.6735  Prec=0.8303  Rec=0.7810 ← best
    t=0.45: Dice=0.8061  IoU=0.6751  Prec=0.8471  Rec=0.7688 ← best
    t=0.50: Dice=0.8056  IoU=0.6745  Prec=0.8619  Rec=0.7562
    t=0.55: Dice=0.8037  IoU=0.6719  Prec=0.8762  Rec=0.7424
    t=0.60: Dice=0.8000  IoU=0.6667  Prec=0.8913  Rec=0.7257
  Best threshold for fold 3: 0.45
 Fold 3 | Best Val Dice: 0.7955 | Best Threshold: 0.45

--- FOLD 4/5 ---
  Train: 439 | Val: 110


...

Ensemble (5 models),   t=0.50, no TTA: 100%|██████████| 25/25 [00:02<00:00, 11.61it/s]



  Ensemble (5 models),   t=0.50, no TTA
    dice        : 0.8467
    iou         : 0.7439
    precision   : 0.8969
    recall      : 0.8121
    f1          : 0.8467


Ensemble (5 models), t=0.60,  5-TTA: 100%|██████████| 25/25 [00:09<00:00,  2.70it/s]


  Ensemble (5 models), t=0.60,  5-TTA
    dice        : 0.7966
    iou         : 0.6800
    precision   : 0.9440
    recall      : 0.7120
    f1          : 0.7966


 ABLATION TABLE — MaxDiceUNet v4
Configuration                                  Dice    IoU   Prec    Rec     F1
----------------------------------------------------------------------------------
Single model (fold 1), t=0.50, no TTA        0.8170 0.7063 0.8548 0.8036 0.8170
Ensemble (5 models),   t=0.50, no TTA        0.8467 0.7439 0.8969 0.8121 0.8467
Ensemble (5 models), t=0.60,  5-TTA          0.7966 0.6800 0.9440 0.7120 0.7966


 PAPER REPORTING NUMBERS
  CV  Dice  : 0.8088 ± 0.0134
  Test Dice : 0.7966
  Test IoU  : 0.6800
  Test F1   : 0.7966
  Threshold : 0.6

 Saving sample predictions...



 All outputs saved to: /kaggle/working
  Files: best_model_fold_1..5.pth | curve_fold_1..5.png | sample_predictions.png
